# Investigating Factors Associated with STAI Lower Than Baseline ($\Delta \text{STAI} < 0$)

### Research Motivation & Context
In psychophysiological studies utilizing wearable sensors (such as WESAD), a common observation is that participants do not always report increased anxiety during stress induction protocols (e.g., TSST), and frequently report **lower state anxiety** during relaxation, amusement, or control tasks relative to their initial baseline (`Base`). 

This notebook shifts our research question from binary stress classification to answering: **What physiological and contextual factors characterize when a participant's State-Trait Anxiety Inventory (STAI) score drops below their personal baseline ($\Delta \text{STAI} < 0$)?**

### Analytical Pipeline Architecture
1. **Questionnaire Parsing & Delta Computation**: Extracts repeated STAI questionnaire responses per condition, establishes individual baseline means, and computes condition-level deltas ($\Delta \text{STAI} = \text{Mean}_{\text{cond}} - \text{Mean}_{\text{Base}}$).
2. **Multimodal Feature Correlation Profiling**: Maps 60-second windowed physiological features (EDA tonic/phasic, ECG/HRV metrics, respiration, and accelerometer movement) against STAI decreases.
3. **Subject-Grouped Predictive Modeling**: Trains classifiers (Random Forest, SVM, XGBoost, KNN) using strict participant-level splitting (`GroupShuffleSplit`) to evaluate generalization without data leakage across participants.

In [1]:
import gc
import pickle
import sys
import warnings
from pathlib import Path
from unittest.mock import MagicMock

import matplotlib
matplotlib.use('Agg')
import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

warnings.simplefilter('ignore')

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'WESAD').exists())
WESAD_DIR = REPO_ROOT / 'WESAD'
OUTPUT_DIR = REPO_ROOT / 'Data' / 'Preprocessed n STAI-labeled'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = REPO_ROOT / 'Models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

SUBJECTS = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]
print('Setup completed:', OUTPUT_DIR)

Setup completed: e:\Rizqy\Kuliah\Skripsi\repo-skripsi\Data\Preprocessed n STAI-labeled


## 1. Questionnaire & STAI Delta Parsing

### Why this step is necessary:
WESAD questionnaire files (`S*_quest.csv`) contain self-reported STAI item responses (values 1–4) and precise timing intervals (`# ORDER`, `# START`, `# END`) for every experimental condition. Because absolute STAI scores vary significantly between individuals due to trait differences, raw scores cannot be directly pooled across participants.

By calculating personal baseline STAI means ($\text{Mean}_{\text{Base}}$) and subtracting them from each condition's mean ($\Delta \text{STAI} = \text{Mean}_{\text{cond}} - \text{Mean}_{\text{Base}}$), we obtain a normalized, baseline-relative metric. Tagging conditions where $\Delta \text{STAI} < 0$ isolates instances of subjective relaxation or anxiety reduction relative to baseline.

In [2]:
def parse_clock_mmss(value):
    text = str(value).strip()
    if not text:
        return np.nan
    if '.' in text:
        minute_text, second_text = text.split('.', 1)
        minutes = int(minute_text)
        seconds = int(second_text)
    else:
        minutes, seconds = int(text), 0
    return minutes * 60 + seconds

def _parse_numeric_fields(parts):
    values = []
    for part in parts:
        text = part.strip()
        if not text:
            values.append(np.nan)
        else:
            try:
                values.append(float(text))
            except ValueError:
                values.append(np.nan)
    return values

def parse_quest_file(subject_id):
    quest_path = WESAD_DIR / f'S{subject_id}' / f'S{subject_id}_quest.csv'
    if not quest_path.exists():
        raise FileNotFoundError(f'Questionnaire file not found: {quest_path}')

    order = start_raw = end_raw = None
    stai_rows = []
    with open(quest_path, 'r', encoding='latin1') as f:
        for line in f:
            parts = [part.strip() for part in line.rstrip('\r\n').split(';')]
            tag = parts[0].strip() if parts else ''
            if tag.startswith('# ORDER'):
                order = [value.strip() for value in parts[1:]]
            elif tag.startswith('# START'):
                start_raw = parts[1:]
            elif tag.startswith('# END'):
                end_raw = parts[1:]
            elif tag.startswith('# STAI'):
                stai_rows.append(_parse_numeric_fields(parts[1:]))

    if order is None or start_raw is None or end_raw is None:
        raise ValueError(f'{quest_path.name}: missing ORDER, START, or END row')

    conditions = []
    for index, condition in enumerate(order):
        if not condition:
            continue
        start_text = start_raw[index].strip() if index < len(start_raw) else ''
        end_text = end_raw[index].strip() if index < len(end_raw) else ''
        start_seconds = parse_clock_mmss(start_text) if start_text else np.nan
        end_seconds = parse_clock_mmss(end_text) if end_text else np.nan
        responses = [row[index] if index < len(row) else np.nan for row in stai_rows]
        valid_responses = [float(val) for val in responses if pd.notna(val)]
        conditions.append({
            'condition': condition,
            'start_seconds': start_seconds,
            'end_seconds': end_seconds,
            'stai_responses': valid_responses,
            'stai_mean': float(np.mean(valid_responses)) if valid_responses else np.nan,
        })
    return conditions

all_records = []
for subject_id in SUBJECTS:
    try:
        conds = parse_quest_file(subject_id)
    except Exception as e:
        continue
    base_items = [c for c in conds if c['condition'].strip().lower() == 'base']
    if not base_items or pd.isna(base_items[0]['stai_mean']):
        continue
    base_mean = base_items[0]['stai_mean']

    for item in conds:
        c_name = item['condition']
        c_mean = item['stai_mean']
        if pd.isna(c_mean):
            continue
        delta = c_mean - base_mean
        all_records.append({
            'subject_id': subject_id,
            'condition': c_name,
            'stai_mean': c_mean,
            'baseline_stai_mean': base_mean,
            'stai_delta': delta,
            'is_lower_than_baseline': int(delta < 0),
        })

df_deltas = pd.DataFrame(all_records)
df_deltas.to_csv(OUTPUT_DIR / 'stai_condition_deltas_summary.csv', index=False)
print('STAI Delta Summary Statistics:')
print(df_deltas['stai_delta'].describe())
print('\nCondition-level breakdown:')
summary_cond = df_deltas.groupby('condition').agg(
    mean_delta=('stai_delta', 'mean'),
    prop_lower=('is_lower_than_baseline', 'mean'),
    count=('stai_delta', 'count')
).reset_index()
display(summary_cond)

STAI Delta Summary Statistics:
count    90.000000
mean     -0.631111
std       0.855339
min      -2.400000
25%      -1.350000
50%      -0.600000
75%       0.000000
max       1.800000
Name: stai_delta, dtype: float64

Condition-level breakdown:


,condition,mean_delta,prop_lower,count
0,Base,0.000000,0.000000,15
1,Fun,-0.360000,0.733333,15
2,Medi 1,-1.226667,0.866667,15
3,Medi 2,-1.266667,0.933333,15
4,TSST,-0.440000,0.533333,15
5,bRead,-0.542857,0.857143,14
6,sRead,0.200000,0.000000,1


## 2. Feature Correlation Analysis

### Why this step is necessary:
Once condition-level STAI drops ($\Delta \text{STAI} < 0$) are identified, we map them back to preprocessed 60-second multimodal physiological windows (`S*_stai.pkl`). This step evaluates whether autonomic nervous system markers—such as electrodermal activity tonic levels (Skin Conductance Level / SCL), heart rate variability (Inter-Beat Interval / IBI), respiration patterns, and physical movement—systematically differ during states of reduced state anxiety compared to non-reduced states.

Pearson point-biserial correlations and group mean comparisons quantify the physiological signature of lowered self-reported anxiety.

In [3]:
pkl_files = sorted(OUTPUT_DIR.glob('S*_stai.pkl'))
dfs = []
for path in pkl_files:
    df = pd.read_pickle(path)
    if 'condition_stai_mean' not in df.columns or 'baseline_stai_mean' not in df.columns:
        continue
    df['stai_delta'] = df['condition_stai_mean'] - df['baseline_stai_mean']
    df['stai_drop'] = (df['stai_delta'] < 0).astype(int)
    dfs.append(df)

combined = pd.concat(dfs, ignore_index=True)
metadata_cols = {
    'binary_anxiety', 'subject_id', 'condition', 'window_start_seconds',
    'window_end_seconds', 'condition_stai_mean', 'baseline_stai_mean',
    'stai_delta', 'stai_drop'
}
feature_cols = [c for c in combined.columns if c not in metadata_cols]

X = combined[feature_cols].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
y = combined['stai_drop']

corrs = []
for col in feature_cols:
    series = X[col]
    if series.nunique() <= 1:
        continue
    corr = series.corr(y, method='pearson')
    corrs.append({
        'feature': col,
        'correlation_with_drop': corr,
        'mean_stai_drop': series[y == 1].mean(),
        'mean_stai_nodrop': series[y == 0].mean(),
    })

df_corr = pd.DataFrame(corrs).sort_values(by='correlation_with_drop', key=abs, ascending=False)
df_corr.to_csv(OUTPUT_DIR / 'stai_drop_feature_correlations.csv', index=False)
print('Top 15 Features Correlated with STAI Decrease (Delta < 0):')
display(df_corr.head(15))

Top 15 Features Correlated with STAI Decrease (Delta < 0):


,feature,correlation_with_drop,mean_stai_drop,mean_stai_nodrop
14,EDA_SCR_min,0.378730,-0.071492,-0.265053
19,EDA_SCL_max,-0.340867,0.168333,1.152276
95,ECG_ibi,0.335193,805.609800,684.312698
16,EDA_SCL_mean,-0.317963,0.092278,0.946879
55,EDA_SCL_C_max,-0.304729,0.235308,0.909269
52,EDA_SCL_C_mean,-0.299656,0.151313,0.758745
18,EDA_SCL_min,-0.299437,0.019255,0.780383
51,EDA_SCR_C_max,-0.293663,0.064500,0.153537
54,EDA_SCL_C_min,-0.290237,0.076304,0.629611
17,EDA_SCL_std,-0.288592,0.044404,0.109924


## 3. Machine Learning Model Training & Evaluation

### Why this step is necessary:
To test whether physiological signals can reliably predict when a participant is experiencing lowered state anxiety relative to baseline, we train supervised classification models (Random Forest, SVM, XGBoost, and KNN).

Crucially, to prevent **subject-wise data leakage** (where high temporal autocorrelation within a participant's data artificially inflates performance), we enforce `GroupShuffleSplit` partitioned strictly by `subject_id`. This ensures that test subjects are entirely unseen during training, providing a rigorous test of real-world generalization.

In [4]:
X = combined[feature_cols].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
y = combined['stai_drop'].astype(int)
groups = combined['subject_id'].astype(int)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, eval_metric='logloss', random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
}

for name, model in models.items():
    print(f'\n--- Training {name} ---')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan
    print(f'{name} Accuracy: {acc:.4f} | ROC AUC: {auc:.4f}')
    print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))
    print('Classification Report:\n', classification_report(y_test, y_pred, zero_division=0))
    joblib.dump(model, MODELS_DIR / f'stai_drop_{name.lower().replace(" ", "_")}_model.joblib')

print('\nAll STAI decrease prediction models trained and evaluated successfully.')


--- Training Random Forest ---
Random Forest Accuracy: 0.4593 | ROC AUC: 0.4701
Confusion Matrix:
 [[  4 194]
 [ 32 188]]
Classification Report:
               precision    recall  f1-score   support

           0       0.11      0.02      0.03       198
           1       0.49      0.85      0.62       220

    accuracy                           0.46       418
   macro avg       0.30      0.44      0.33       418
weighted avg       0.31      0.46      0.34       418


--- Training SVM ---
SVM Accuracy: 0.4904 | ROC AUC: 0.6400
Confusion Matrix:
 [[ 27 171]
 [ 42 178]]
Classification Report:
               precision    recall  f1-score   support

           0       0.39      0.14      0.20       198
           1       0.51      0.81      0.63       220

    accuracy                           0.49       418
   macro avg       0.45      0.47      0.41       418
weighted avg       0.45      0.49      0.43       418


--- Training XGBoost ---
XGBoost Accuracy: 0.5000 | ROC AUC: 0.4502
Con